In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from losses.final_lossv2 import InverseDistanceBoundaryDiceLoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.unets_helper_functions import (
    set_seed,
    save_checkpoint,
    psv2_final_PatchDataset_cbam,
    evaluate_full_volume_cbam,
    compute_per_class_dice
    
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 8
FOREGROUND_PROB = 0.35
GLOBAL_PROB = 0.3
NUM_WORKERS = 2
train_dataset = psv2_final_PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        global_prob=GLOBAL_PROB,
                        augment=True)

val_dataset = psv2_final_PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        global_prob=GLOBAL_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [6]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24 
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24
).to(device)

model_path = "../experiments/psv2/psv2_dice_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\dhanu\AppData\Local\Temp\ipykernel_1380\10116252.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [8]:
import nibabel as nib

def tta_predict(model, patch):
    preds = []

    for flip_dim in [None, 2, 3, 4]:
        if flip_dim is not None:
            patch_flip = torch.flip(patch, dims=[flip_dim])
        else:
            patch_flip = patch

        logits = model(patch_flip)[0]

        if flip_dim is not None:
            logits = torch.flip(logits, dims=[flip_dim])

        preds.append(torch.softmax(logits, dim=1))

    return torch.mean(torch.stack(preds), dim=0)

def get_gaussian_weight_map(patch_size):
    import numpy as np

    ranges = [np.linspace(-1, 1, s) for s in patch_size]
    z, y, x = np.meshgrid(*ranges, indexing="ij")

    dist = z**2 + y**2 + x**2
    gaussian = np.exp(-dist / 0.3)

    gaussian = gaussian / np.max(gaussian)
    return torch.tensor(gaussian, dtype=torch.float32)


def final_sliding_window_cbam(model, image, patch_size=96, stride=48, device="cuda"):
    model.eval()

    image_t = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    _, _, D, H, W = image_t.shape
    num_classes = 7

    # CPU accumulation (safe for memory)
    output     = torch.zeros((1, num_classes, D, H, W), dtype=torch.float32)
    weight_map = torch.zeros_like(output)

    gaussian = get_gaussian_weight_map((patch_size, patch_size, patch_size))
    gaussian_gpu = gaussian.unsqueeze(0).unsqueeze(0).to(device)
    gaussian_cpu = gaussian.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        z_steps = sorted(set(list(range(0, max(1, D - patch_size), stride)) + [max(0, D - patch_size)]))
        y_steps = sorted(set(list(range(0, max(1, H - patch_size), stride)) + [max(0, H - patch_size)]))
        x_steps = sorted(set(list(range(0, max(1, W - patch_size), stride)) + [max(0, W - patch_size)]))

        print(f"Volume: {D}x{H}x{W} | Patches: {len(z_steps)*len(y_steps)*len(x_steps)} | Stride: {stride}")

        for z in z_steps:
            for y in y_steps:
                for x in x_steps:

                    patch = image_t[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size]

                    # ---- Forward ----
                    # logits = model(patch)[0]   # deep supervision → take main output
                    probs = tta_predict(model, patch)

                    # ---- Weight with Gaussian ----
                    probs = probs * gaussian_gpu

                    # ---- Move to CPU once ----
                    probs_cpu = probs.cpu()

                    output[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += probs_cpu
                    weight_map[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += gaussian_cpu

    # ---- Normalize ----
    output = output / weight_map.clamp(min=1e-6)

    # ---- Argmax ----
    pred = torch.argmax(output, dim=1).squeeze().numpy()

    return pred


def final_evaluate_full_volume_cbam(model, cases, images_dir, labels_dir,
                             patch_size=96, stride=48, device="cuda"):

    model.eval()
    all_dices = []

    for case in cases:

        image = nib.load(f"{images_dir}/{case}.nii.gz").get_fdata(dtype=np.float32)
        label = nib.load(f"{labels_dir}/{case}.nii.gz").get_fdata().astype(np.uint8)

        pred = final_sliding_window_cbam(
            model,
            image,
            patch_size=patch_size,
            stride=stride,
            device=device
        )

        print(f"\nCase: {case}")
        print("Pred labels:", np.unique(pred))
        print("GT labels:", np.unique(label))

        case_dices = []

        for cls in range(1, 7):  # ignore background

            pred_cls = (pred == cls)
            label_cls = (label == cls)

            intersection = np.sum(pred_cls & label_cls)
            union = np.sum(pred_cls) + np.sum(label_cls)

            # ---- SAFE Dice ----
            if union == 0:
                dice = 1.0  # both empty → perfect
            else:
                dice = (2 * intersection) / union

            case_dices.append(dice)

        all_dices.append(case_dices)

        print(f"Dice: {np.round(case_dices, 4)}")

    all_dices = np.array(all_dices)

    mean_dice = np.mean(all_dices, axis=0)
    std_dice  = np.std(all_dices, axis=0)

    print("\n===== FINAL RESULTS =====")
    print("Mean Dice:", np.round(mean_dice, 4))
    print("Std Dice :", np.round(std_dice, 4))

    return mean_dice, std_dice

In [ ]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9213 0.8143 0.7939 0.8269 0.8372 0.9064]
Volume: 243x383x383 | Patches: 245 | Stride: 48

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9161 0.7477 0.719  0.8051 0.8576 0.8684]
Volume: 264x333x333 | Patches: 180 | Stride: 48

Case: case_22
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9043 0.8311 0.8465 0.8085 0.8398 0.901 ]
Volume: 248x395x395 | Patches: 320 | Stride: 48

Case: case_14
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.8983 0.7409 0.7426 0.8192 0.8408 0.7814]
Volume: 268x413x413 | Patches: 320 | Stride: 48

Case: case_20
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9093 0.7923 0.6016 0.7783 0.7596 0.8053]
Volume: 244x415x415 | Patches: 320 | Stride: 48

Case: case_17
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.8767 0.827  0.7282 0.814  0

In [11]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    stride=32,
    device=device
)

Volume: 272x357x357 | Patches: 700 | Stride: 32

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9229 0.8109 0.8199 0.832  0.8395 0.9083]
Volume: 243x383x383 | Patches: 600 | Stride: 32

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9182 0.7468 0.7148 0.8059 0.8563 0.8772]
Volume: 264x333x333 | Patches: 567 | Stride: 32

Case: case_22
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9103 0.8354 0.8494 0.8103 0.8395 0.8925]
Volume: 248x395x395 | Patches: 726 | Stride: 32

Case: case_14
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.899  0.7441 0.7305 0.8201 0.8354 0.7849]
Volume: 268x413x413 | Patches: 847 | Stride: 32


KeyboardInterrupt: 

In [9]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    stride=32,
    device=device
)

Volume: 272x357x357 | Patches: 700 | Stride: 32

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9245 0.8038 0.8414 0.849  0.8393 0.9091]
Volume: 243x383x383 | Patches: 600 | Stride: 32

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9229 0.7481 0.7213 0.7915 0.8543 0.8767]
Volume: 264x333x333 | Patches: 567 | Stride: 32


KeyboardInterrupt: 